In [1]:
import os
import copy
import pickle
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
from sklearn.model_selection import train_test_split


# =========================
# 0) Utils
# =========================
def ensure_dir(p: str) -> str:
    os.makedirs(p, exist_ok=True)
    return p

def load_pkl(path: str) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)

def save_pkl(obj: Any, path: str):
    ensure_dir(os.path.dirname(path))
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)

def assert_same_len(a: List, b: List, name_a: str, name_b: str):
    if len(a) != len(b):
        raise ValueError(f"[align] length mismatch: {name_a}={len(a)} != {name_b}={len(b)}")

def check_edge_index(edge_index, E: int, name: str):
    """
    edge_index 支持:
      - [src_list, dst_list]
      - np.ndarray shape (2, M)
    """
    if isinstance(edge_index, list) and len(edge_index) == 2:
        src, dst = edge_index
        mx = max(max(src), max(dst)) if (len(src) and len(dst)) else -1
    else:
        arr = np.asarray(edge_index)
        if arr.ndim != 2 or arr.shape[0] != 2:
            raise ValueError(f"[edge] {name} bad shape: {arr.shape}, expect (2,M) or [2 lists]")
        mx = int(arr.max()) if arr.size else -1

    if mx >= E:
        raise ValueError(
            f"[edge] {name} edge id out of range: max_id={mx} but E={E}. "
            f"你的 ent_names 长度和 edge_index 的索引体系不一致。"
        )

def stack_time_series_list(x_list: List[np.ndarray], name: str) -> np.ndarray:
    """
    x_list: list of np.ndarray shape (T_window, D)
    return: (N, T_window, D) float32
    """
    if len(x_list) == 0:
        raise ValueError(f"{name} is empty")
    arr = np.stack([np.asarray(x) for x in x_list], axis=0).astype(np.float32)
    if arr.ndim != 3:
        raise ValueError(f"{name} stacked shape={arr.shape}, expect (N,T,D)")
    return arr

def stack_y_list(y_list: List[np.ndarray], E: int, name: str) -> np.ndarray:
    """
    y_list: list of np.ndarray shape (E,)
    return: (N,E) int64
    """
    if len(y_list) == 0:
        raise ValueError(f"{name} is empty")
    arr = np.stack([np.asarray(y).reshape(-1) for y in y_list], axis=0).astype(np.int64)
    if arr.ndim != 2 or arr.shape[1] != E:
        raise ValueError(f"{name} stacked shape={arr.shape}, expect (N,E)=(*,{E})")
    return arr



def infer_ent_fault_type_index(fault_type_list: List[str]) -> Dict[str, Tuple[int, int]]:
    node_idx = [i for i, ft in enumerate(fault_type_list) if ft.startswith("node_")]
    svcpod_idx = [i for i, ft in enumerate(fault_type_list) if ft.startswith("container_")]

    if not node_idx or not svcpod_idx:
        raise ValueError("Need both node_* and container_* fault types.")

    if node_idx != list(range(min(node_idx), max(node_idx) + 1)):
        raise ValueError(f"node fault types are not contiguous: {node_idx}")
    if svcpod_idx != list(range(min(svcpod_idx), max(svcpod_idx) + 1)):
        raise ValueError(f"container fault types are not contiguous: {svcpod_idx}")

    return {
        "node": (min(node_idx), max(node_idx) + 1),
        "service": (min(svcpod_idx), max(svcpod_idx) + 1),
        "pod": (min(svcpod_idx), max(svcpod_idx) + 1),
    }


def infer_ent_fault_type_weight(
    y_train_valid: List[np.ndarray],
    ent_type_index: Dict[str, Tuple[int, int]],
    num_fault_types: int,
) -> Dict[str, List[int]]:
    y_arr = np.stack([np.asarray(y).reshape(-1) for y in y_train_valid], axis=0)
    out = {}
    for ent_type, (s, e) in ent_type_index.items():
        sub = y_arr[:, s:e]
        counts = []
        for ft_id in range(1, num_fault_types + 1):
            counts.append(int(np.count_nonzero(sub == ft_id)))
        out[ent_type] = counts
    return out


def trim_ent_fault_type_weight(
    weight_full: Dict[str, List[int]],
    ent_fault_type_index: Dict[str, Tuple[int, int]],
) -> Dict[str, List[int]]:
    out = {}
    for ent_type, (s, e) in ent_fault_type_index.items():
        out[ent_type] = [max(1, x) for x in weight_full[ent_type][s:e]]
    return out


def build_fault_keyword_map(fault_type_list: List[str]) -> Dict[str, List[str]]:
    out = {}
    for ft in fault_type_list:
        kws = []
        if "cpu" in ft:
            kws += ["cpu", "load", "throttled"]
        if "memory" in ft or "mem" in ft:
            kws += ["mem", "memory", "rss", "working_set", "cache", "usable"]
        if "disk" in ft or "io" in ft:
            kws += ["disk", "io", "read", "write", "await", "svctm", "util", "fs"]
        if "network" in ft:
            kws += ["network", "request", "response", "transmit", "receive", "bytes", "messages", "packet", "latency"]
        if "process" in ft or "abort" in ft:
            kws += ["process", "thread", "error", "exception", "abort", "killed"]

        seen = set()
        dedup = []
        for k in kws:
            if k not in seen:
                seen.add(k)
                dedup.append(k)
        out[ft] = dedup
    return out


def infer_fault_type_related_o11y_names(
    fault_type_list: List[str],
    modal_obj: Dict[str, Any],
) -> Dict[str, List[str]]:
    all_names = []
    if "metric" in modal_obj:
        all_names.extend(modal_obj["metric"]["metric_names"])
    if "trace" in modal_obj:
        all_names.extend(modal_obj["trace"]["trace_names"])
    if "log" in modal_obj:
        all_names.extend(modal_obj["log"]["log_names"])

    keyword_map = build_fault_keyword_map(fault_type_list)
    out = {}
    for ft in fault_type_list:
        kws = keyword_map[ft]
        matched = []
        for name in all_names:
            low = str(name).lower()
            if any(k in low for k in kws):
                matched.append(name)
        out[ft] = matched
    return out




# =========================
# 1) DatasetHandler-like (OpenSource behavior)
# =========================
class DatasetHandlerLikeOpenSource:
    """
    目标：贴近论文开源 DatasetHandler.split_and_save_dataset 的使用方式：
    - 输入：modal_data(按 split 的 list)、ent_edge_index(按 split 的 list)、label_obj(按 split 的 y list)
    - 只对 train_valid 做 train/valid 切分
    - y 使用 multi-class id 格式：(N,E) int64，不做 multi-hot 展开
    """
    @staticmethod
    def split_and_save_dataset(
        modal_type_list: List[str],
        modal_data: Dict[str, Dict[str, List[np.ndarray]]],   # modal -> {train_valid/test: list[(T,D)]}
        ent_edge_index: Dict[str, List[Any]],                 # {train_valid/test: list[edge_index]}
        label_obj: Dict[str, Any],                            # 含 y / entity_type / template / cmdb_id / root_cause_type
        meta_data: Dict[str, Any],
        save_file_path: str,
        valid_ratio: float = 0.2,
        random_state: int = 409,
        shuffle: bool = True,
        multi_class_label_format: bool = True,
        num_of_fault_types: int = 15,   # 你的标签 id 是 1..K，这里就填 K（不含0）
        save_extra_fields: bool = True,
        extra_keys: Optional[List[str]] = None,
    ):
        meta = copy.deepcopy(meta_data)
        meta["multi_class_label_format"] = bool(multi_class_label_format)
        meta["num_of_fault_types"] = int(num_of_fault_types)

        if extra_keys is None:
            extra_keys = ["entity_type", "template", "cmdb_id", "root_cause_type"]

        # ---- 对齐检查 ----
        '''
            [
                np.array(E,),
                np.array(E,),
                ...
            ]
            每个元素array([0,0,0,2,0,0,0,0,...])
        '''
        y_tv = label_obj["y"]["train_valid"]
        y_te = label_obj["y"]["test"]
        for m in modal_type_list:
            assert_same_len(modal_data[m]["train_valid"], y_tv, f"{m}.train_valid", "y.train_valid")
            assert_same_len(modal_data[m]["test"], y_te, f"{m}.test", "y.test")
        assert_same_len(ent_edge_index["train_valid"], y_tv, "edge.train_valid", "y.train_valid")
        assert_same_len(ent_edge_index["test"], y_te, "edge.test", "y.test")

        if save_extra_fields:
            for k in extra_keys:
                if k not in label_obj:
                    raise ValueError(f"extra field missing in label_obj: {k}")
                assert_same_len(label_obj[k]["train_valid"], y_tv, f"{k}.train_valid", "y.train_valid")
                assert_same_len(label_obj[k]["test"], y_te, f"{k}.test", "y.test")

        # ---- 组 pack：按 sklearn train_test_split 的参数顺序打包 ----
        pack_tv: List[Any] = []
        pack_te: List[Any] = []

        '''
            List len = E(56)
            [array1, array2, ...]
            每个元素维度为(T, num_features)
        '''
        for m in modal_type_list:
            pack_tv.append(modal_data[m]["train_valid"])
            pack_te.append(modal_data[m]["test"])

        '''
            edge_index_window1 = [src, dst]
            src = [s1, s2, s3, ...]
            dst = [d1, d2, d3, ...]
        '''
        pack_tv.append(ent_edge_index["train_valid"])
        pack_te.append(ent_edge_index["test"])

        '''
            [
                np.array(E,),
                np.array(E,),
                ...
            ]
            每个元素array([0,0,0,2,0,0,0,0,...])
        '''        
        pack_tv.append(y_tv)
        pack_te.append(y_te)

        if save_extra_fields:
            for k in extra_keys:
                pack_tv.append(label_obj[k]["train_valid"])
                pack_te.append(label_obj[k]["test"])

        # ---- split train_valid -> train/valid ----
        split_out = train_test_split(
            *tuple(pack_tv),
            test_size=valid_ratio,
            random_state=random_state,
            shuffle=shuffle,
        )
        train_pack = split_out[::2]
        valid_pack = split_out[1::2]
        test_pack = pack_te

        # ---- 输出 ----
        result: Dict[str, Any] = {"meta_data": meta, "data": {}}
        E = len(meta["ent_names"])

        def _write_split(split_name: str, pack: List[Any]):
            # modal
            for i, m in enumerate(modal_type_list):
                result["data"][f"x_{m}_{split_name}"] = stack_time_series_list(pack[i], f"x_{m}_{split_name}")

            # edge
            edge_idx = len(modal_type_list)
            result["data"][f"ent_edge_index_{split_name}"] = pack[edge_idx]

            # y: list[(E,)] -> (N,E)
            y_idx = len(modal_type_list) + 1
            result["data"][f"y_{split_name}"] = stack_y_list(pack[y_idx], E, f"y_{split_name}")

            # extra
            if save_extra_fields:
                base = len(modal_type_list) + 2
                for t, k in enumerate(extra_keys):
                    extra_list = pack[base + t]
                    result["data"][f"{k}_{split_name}"] = np.array(extra_list, dtype=object)

        _write_split("train", train_pack)
        _write_split("valid", valid_pack)
        _write_split("test", test_pack)

        '''
            data =
            {
              # metric
              "x_metric_train": (N_train, T_window, F_metric),
              "x_metric_valid": (N_valid, T_window, F_metric),
              "x_metric_test":  (N_test,  T_window, F_metric),
            
              # trace
              "x_trace_train": (N_train, T_window, F_trace),
              "x_trace_valid": (N_valid, T_window, F_trace),
              "x_trace_test":  (N_test,  T_window, F_trace),
            
              # log
              "x_log_train": (N_train, T_window, F_log),
              "x_log_valid": (N_valid, T_window, F_log),
              "x_log_test":  (N_test,  T_window, F_log),
            
              # 图结构
              "ent_edge_index_train": list[N_train],
              "ent_edge_index_valid": list[N_valid],
              "ent_edge_index_test":  list[N_test],
            
              # 标签
              "y_train": (N_train, E),
              "y_valid": (N_valid, E),
              "y_test":  (N_test, E),
            
              # extra fields
              # 例如 ["service","pod","node",...]
              "entity_type_train": array(N_train),
              "entity_type_valid": array(N_valid),
              "entity_type_test":  array(N_test),

              # 例如["checkoutservice","frontend",...]
              "template_train": array(N_train),
              "template_valid": array(N_valid),
              "template_test":  array(N_test),

              # 例如["checkoutservice","node-3",...]
              "cmdb_id_train": array(N_train),
              "cmdb_id_valid": array(N_valid),
              "cmdb_id_test":  array(N_test),

              # 例如["network","cpu","memory",...]
              "root_cause_type_train": array(N_train),
              "root_cause_type_valid": array(N_valid),
              "root_cause_type_test":  array(N_test),
            }
        '''
        save_pkl(result, save_file_path)
        print(f"[OK] saved -> {save_file_path}")
        print("     y_train shape:", result["data"]["y_train"].shape, "(expect (N,E))")


# =========================
# 2) Generator Config
# =========================
@dataclass
class AIOpsDatasetGeneratorConfig:
    temp_data_storage: str = "../../temp_data/aiops22"
    window_size_list: Optional[List[int]] = None
    modal_types: Optional[List[str]] = None
    valid_ratio: float = 0.2
    seed: int = 409

    node_list: Optional[List[str]] = None
    service_list: Optional[List[str]] = None
    pod_list: Optional[List[str]] = None
    all_entity_list: Optional[List[str]] = None

    fault_type_list: Optional[List[str]] = None  # 例如 ["network","io","cpu","process","memory"]


# =========================
# 3) AIOpsDatasetGenerator (merge)
# =========================
class AIOpsDatasetGeneratorOpenSourceLike:
    """
    依赖你已经生成好的中间文件：
      dataset/metric/metric_window_size_{w}.pkl
      dataset/trace/trace_window_size_{w}.pkl
      dataset/log/log_window_size_{w}.pkl
      dataset/ent_edge_index/ent_edge_index_window_size_{w}.pkl
      dataset/time_interval_and_label/time_interval_window_size_{w}.pkl   <-- 你新的 label
    """
    def __init__(self, cfg: AIOpsDatasetGeneratorConfig):
        if cfg.window_size_list is None:
            cfg.window_size_list = [5, 10, 15]
        if cfg.modal_types is None:
            cfg.modal_types = ["metric", "trace", "log"]
        if cfg.node_list is None or cfg.service_list is None or cfg.pod_list is None:
            raise ValueError("You must provide node_list/service_list/pod_list")
        if cfg.all_entity_list is None:
            cfg.all_entity_list = cfg.node_list + cfg.service_list + cfg.pod_list
        if cfg.fault_type_list is None:
            raise ValueError("You must provide fault_type_list")

        self.cfg = cfg
        self.base = cfg.temp_data_storage
        self.dataset_dir = os.path.join(self.base, "dataset")

        self.metric_dir = os.path.join(self.dataset_dir, "metric")
        self.trace_dir  = os.path.join(self.dataset_dir, "trace")
        self.log_dir    = os.path.join(self.dataset_dir, "log")
        self.edge_dir   = os.path.join(self.dataset_dir, "ent_edge_index")
        self.label_dir  = os.path.join(self.dataset_dir, "time_interval_and_label")
        self.out_dir    = ensure_dir(os.path.join(self.dataset_dir, "merge_multimodal"))

        # meta 基础：与开源 DatasetGenerator.meta_data 同类型字段
        self.ent_types = ["node", "service", "pod"]
        self.ent_names = list(cfg.all_entity_list)
        self.E = len(self.ent_names)

        n_node = len(cfg.node_list)
        n_svc  = len(cfg.service_list)
        n_pod  = len(cfg.pod_list)

        self.ent_type_index = {
            "node": (0, n_node),
            "service": (n_node, n_node + n_svc),
            "pod": (n_node + n_svc, n_node + n_svc + n_pod),
        }
        if self.ent_type_index["pod"][1] != self.E:
            raise ValueError("[meta] ent_type_index does not cover all entities; check list lengths.")

    def _load_modal(self, modal: str, window_size: int) -> Dict[str, Any]:
        if modal == "metric":
            return load_pkl(os.path.join(self.metric_dir, f"metric_window_size_{window_size}.pkl"))
        if modal == "trace":
            return load_pkl(os.path.join(self.trace_dir, f"trace_window_size_{window_size}.pkl"))
        if modal == "log":
            return load_pkl(os.path.join(self.log_dir, f"log_window_size_{window_size}.pkl"))
        raise ValueError(f"Unknown modal: {modal}")

    def _load_edge_index(self, window_size: int) -> Dict[str, Any]:
        return load_pkl(os.path.join(self.edge_dir, f"ent_edge_index_window_size_{window_size}.pkl"))

    def _load_label_obj(self, window_size: int) -> Dict[str, Any]:
        return load_pkl(os.path.join(self.label_dir, f"time_interval_window_size_{window_size}.pkl"))

    
    def _build_meta(self, modal_types: List[str], modal_obj: Dict[str, Any], label_obj, window_size: int) -> Dict[str, Any]:
        meta = {
            "modal_types": list(modal_types),
            "ent_types": list(self.ent_types),
            "ent_names": list(self.ent_names),
            "ent_type_index": dict(self.ent_type_index),

            "ent_features": {m: [] for m in modal_types},
            "max_ent_feature_num": {et: {m: 0 for m in modal_types} for et in self.ent_types},
            "o11y_names": {m: [] for m in modal_types},
            "o11y_length": {m: 0 for m in modal_types},

            "fault_type_list": list(self.cfg.fault_type_list),

            "window_size": int(window_size),
        }

        # names/features
        for m in modal_types:
            obj = modal_obj[m]
            '''
                entity_features:
                [
                    ('node-1', (0,24)),
                    ('node-2', (24,48)),
                    ...
                    ('adservice', (144,160)),
                    ...
                    ('adservice-0', (320,340)),
                    ...
                ]

                metric_names/trace_names/log_names:
                [
                    "node-1/CPU/system.load.1",
                    "node-1/CPU/system.load.5",
                    ...
                    "node-2/CPU/system.load.1",
                    ...
                    "adservice/Request/rr.grpc",
                    ...
                    "adservice-0/CPU/kpi_container_cpu_usage_seconds",
                    ...
                ]
            '''
            if m == "metric":
                meta["ent_features"][m] = obj["entity_features"]
                meta["o11y_names"][m] = obj["metric_names"]
                meta["o11y_length"][m] = len(obj["metric_names"])
            elif m == "trace":
                meta["ent_features"][m] = obj["entity_features"]
                meta["o11y_names"][m] = obj["trace_names"]
                meta["o11y_length"][m] = len(obj["trace_names"])
            elif m == "log":
                meta["ent_features"][m] = obj["entity_features"]
                meta["o11y_names"][m] = obj["log_names"]
                meta["o11y_length"][m] = len(obj["log_names"])

        # max feature num per ent_type per modal
        for ent_type, (s, e) in self.ent_type_index.items():
            for m in modal_types:
                ent_feat = meta["ent_features"][m]
                if len(ent_feat) < e:
                    raise ValueError(f"[meta] entity_features too short for {m}: {len(ent_feat)} < {e}")
                for i in range(s, e):
                    start, end = ent_feat[i][1]
                    n_feat = int(end) - int(start)
                    meta["max_ent_feature_num"][ent_type][m] = max(meta["max_ent_feature_num"][ent_type][m], n_feat)
      
        # 1) 推导 ent_fault_type_index
        meta["ent_fault_type_index"] = infer_ent_fault_type_index(self.cfg.fault_type_list)
        
        # 2) 推导 ent_fault_type_weight（先全长统计，再按 ent_fault_type_index 截断）
        weight_full = infer_ent_fault_type_weight(
            y_train_valid=label_obj["y"]["train_valid"],
            ent_type_index=self.ent_type_index,
            num_fault_types=len(self.cfg.fault_type_list),
        )
        meta["ent_fault_type_weight"] = trim_ent_fault_type_weight(
            weight_full,
            meta["ent_fault_type_index"],
        )
        
        # 3) 推导 fault_type_related_o11y_names
        meta["fault_type_related_o11y_names"] = infer_fault_type_related_o11y_names(
            self.cfg.fault_type_list,
            modal_obj,
        )

        return meta

    def generate(self, save_extra_fields: bool = True):
        modal_types = self.cfg.modal_types
        K = len(self.cfg.fault_type_list)  # 你的 fault_type_id 是 1..K

        for window_size in self.cfg.window_size_list:
            print("\n" + "=" * 80)
            print(f"[GEN] window_size={window_size}")

            # load
            '''
                log:
                    log_data = {
                        "train_valid": [array1, array2, ...],
                        "test":        [array1, array2, ...]
                    }
                    每个array形状为shape = (T_window, F)，其中F = 总日志特征数
                    ================================================================
                    entity_features 每个实体对应的列区间
                    [
                        ("node-1", (0, 0)),
                        ...
                        ("adservice", (0, 0)),   # service占位
                        ...
                        ("adservice-0", (0, 15)),
                        ("adservice-1", (15, 30)),
                        ...
                    ]
                    =================================================================
                    log_names 每一列的真实含义，长度为F
                    [
                      "adservice-0; Java; <log pattern 3>",
                      "adservice-0; Java; <log pattern 7>",
                      ...
                      "adservice-0; error",
                      ...
                    ]
                =======================================================================
                =======================================================================
                =======================================================================
                trace:
                    trace_data = {
                        "train_valid": [array1, array2, ...],
                        "test":        [array1, array2, ...]
                    }
                    每个 array 形状为(T_window, F)，F为所有 pod 的 trace 特征数量
                    =================================================================
                    entity_features为每个实体（node / service / pod）对应特征矩阵的列索引区间
                    [
                      ("node-1", (0,0)),
                      ...
                      ("adservice", (0,0)),
                      ...
                      ("adservice-0", (0,6)),
                      ("adservice-1", (6,12)),
                      ...
                    ]
                    =================================================================
                    trace_names为特征名
                    [
                      "<intensity>; cmdb_id: adservice-0; type: upstream",
                      "<duration>; cmdb_id: adservice-0; type: upstream",
                      ...
                    ]
                =======================================================================
                =======================================================================
                =======================================================================
                metric:
                    metric_data = {
                        "train_valid": [
                            window_matrix_1,
                            window_matrix_2,
                            ...
                        ],
                        "test": [
                            window_matrix_1,
                            window_matrix_2,
                            ...
                        ]
                    }
                    每一个 window_matrix形状为(T_window, feature_dim)
                    =================================================================
                    entity_features = [
                        (entity_name, (start_index, end_index)),
                        ...
                    ]
    
                    [
                        ('node-1', (0,24)),
                        ('node-2', (24,48)),
                        ...
                        ('adservice', (144,160)),
                        ...
                        ('adservice-0', (320,340)),
                        ...
                    ]
                    =================================================================
                    metric_names为指标名
                    [
                        "node-1/CPU/system.load.1",
                        "node-1/CPU/system.load.5",
                        ...
                        "node-2/CPU/system.load.1",
                        ...
                        "adservice/Request/rr.grpc",
                        ...
                        "adservice-0/CPU/kpi_container_cpu_usage_seconds",
                        ...
                    ]                

            '''
            modal_obj = {m: self._load_modal(m, window_size) for m in modal_types}
            '''
                {
                  "train_valid": [
                      edge_index_window1,
                      edge_index_window2,
                      ...
                  ],
                  "test": [
                      edge_index_window1,
                      ...
                  ]
                }
                edge_index_window1 = [src, dst]
                src = [s1, s2, s3, ...]
                dst = [d1, d2, d3, ...]
            '''
            edge_obj  = self._load_edge_index(window_size)
            '''
                {
                "time_interval": {
                  "train_valid": [...],
                  "test": [...]
                },
                
                "y": {
                  "train_valid": [...],
                  "test": [...]
                },
                
                "entity_type": {
                  "train_valid": [...],
                  "test": [...]
                },
                
                "template": {
                  "train_valid": [...],
                  "test": [...]
                },
                
                "cmdb_id": {
                  "train_valid": [...],
                  "test": [...]
                },
                
                "root_cause_type": {
                  "train_valid": [...],
                  "test": [...]
                }
                }
                ====================================================
                time_interval是一个列表
                [
                ["2022-03-20","cloudbed-1",1647763200,1647763800],
                ["2022-03-20","cloudbed-1",1647764100,1647764700],
                ]
                ====================================================
                y 是元素标签列表
                [
                np.array(E,),
                np.array(E,),
                ...
                ]
                每个元素array([0,0,0,2,0,0,0,0,...])
                ====================================================
                entity_type为故障实体类型
                ["service","pod","node","service",...]
                ====================================================
                template为大的归类:微服务/node
                ["checkoutservice","node",...]
                ====================================================
                cmdb_id为发生故障的实体
                ["checkoutservice","node-3",...]
                ====================================================
                root_cause_type为根因类型
                ["network","cpu","io",...]
                ====================================================
                所有数据长度都相等
            '''
            label_obj = self._load_label_obj(window_size)

            # modal_data dict: modal -> split -> list[(T,D)]
            modal_data: Dict[str, Dict[str, List[Any]]] = {}
            for m in modal_types:

                # {
                #     "train_valid": [
                #         window_matrix_1,
                #         window_matrix_2,
                #         ...
                #     ],
                #     "test": [
                #         window_matrix_1,
                #         window_matrix_2,
                #         ...
                #     ]
                # }
                # 每一个 window_matrix形状为(T_window, feature_dim)

                if m == "metric":
                    modal_data[m] = modal_obj[m]["metric_data"]

                # metric_data = {
                #     "train_valid": [
                #         window_matrix_1,
                #         window_matrix_2,
                #         ...
                #     ],
                #     "test": [
                #         window_matrix_1,
                #         window_matrix_2,
                #         ...
                #     ]
                # }
                # 每一个 window_matrix形状为(T_window, feature_dim)

                elif m == "trace":
                    modal_data[m] = modal_obj[m]["trace_data"]

                # log_data = {
                #     "train_valid": [array1, array2, ...],
                #     "test":        [array1, array2, ...]
                # }
                # 每个array形状为shape = (T_window, F)，其中F = 总日志特征数

                elif m == "log":
                    modal_data[m] = modal_obj[m]["log_data"]

            # edge_split dict
            edge_split = edge_obj["ent_edge_index"] if ("ent_edge_index" in edge_obj and isinstance(edge_obj["ent_edge_index"], dict)) else edge_obj

            # sanity: edge id range
            check_edge_index(edge_split["train_valid"][0], self.E, "edge.train_valid[0]")
            check_edge_index(edge_split["test"][0], self.E, "edge.test[0]")

            # meta
            '''
            {
              # 模态
              "modal_types": ["metric","trace","log"],
              ====================================================
              # 实体类型
              "ent_types": ["node","service","pod"],
              ====================================================
              # 实体名称列表
              "ent_names": [...56 entities...],
              ====================================================            
              # 每种实体类型在列表中的 index 范围
              "ent_type_index": {
                  "node": (0,6),
                  "service": (6,16),
                  "pod": (16,56)
              },
              ====================================================            
              # 每个实体对应的特征范围
              "ent_features": {
                  "metric": [],
                  "trace": [],
                  "log": []
              },
              [
                  ('node-1', (0,24)),
                  ('node-2', (24,48)),
                  ...
                  ('adservice', (144,160)),
                  ...
                  ('adservice-0', (320,340)),
                  ...
              ]
              ====================================================            
              # 每种实体类型最大的特征种类/数量
              "max_ent_feature_num": {
                  "node": {trace:n, metric:m, log:r},
                  "service": {trace:n, metric:m, log:r},
                  "pod": {trace:n, metric:m, log:r}
              },
              ====================================================            
              # 每个模态的 feature 名称
              "o11y_names": {
                  "metric": [...],
                  "trace": [...],
                  "log": [...]
              },
              metric_names/trace_names/log_names:
              [
                  "node-1/CPU/system.load.1",
                  "node-1/CPU/system.load.5",
                  ...
                  "node-2/CPU/system.load.1",
                  ...
                  "adservice/Request/rr.grpc",
                  ...
                  "adservice-0/CPU/kpi_container_cpu_usage_seconds",
                  ...
               ]
              ====================================================            
              # 每个模态的特征数量
              "o11y_length": {
                  "metric": 100,
                  "trace": 40,
                  "log": 200
              },
              ====================================================            
              # 故障类型
              "fault_type_list": [
                    "node_cpu_failure",
                    "node_disk_read_io_load",
                    "node_memory_load",
                    "node_disk_space_load",
                    "node_disk_write_io_load",
                    "node_cpu_spike",
                    "container_network_packet_duplicate",
                    "container_write_io_load",
                    "container_read_io_load",
                    "container_cpu_load",
                    "container_network_latency",
                    "container_process_abort",
                    "container_network_packet_loss",
                    "container_memory_load",
                    "container_network_packet_corruption",
                ],
            
              # 当前窗口大小
              "window_size": 5,

              "multi_class_label_format":True,

              "num_of_fault_types" 15, 
              ====================================================
              "ent_fault_type_index"
              {
                  "node": (0,6),
                  "service": (6,15),
                  "pod": (6,15),
              }
              ====================================================
              "ent_fault_type_weight" 统计不同实体中,不同错误出现的次数,解决样本不平衡问题
              {
               "node": [12,8,20,9,5,11],
            
               "service": [32,45,29,50,21,17,40,31,28],
            
               "pod": [120,180,150,200,90,110,170,140,130]
              }
              ====================================================
             "fault_type_related_o11y_names"
             根据 fault type 名称里的关键词，自动从所有观测特征（metric / trace / log）里找出与该 fault type 相关的特征名
              {
                   "node_cpu_failure":
                   [
                         "node-1/CPU/system.load.1",
                         "node-2/CPU/system.load.1",
                         ...
                   ],
                
                   "container_network_latency":
                   [
                         "frontend/Request/latency",
                         "checkoutservice/Request/latency",
                         ...
                   ]
              }
            '''
            meta = self._build_meta(modal_types=modal_types, modal_obj=modal_obj, label_obj=label_obj, window_size=window_size)

            # save
            out_path = os.path.join(self.out_dir, f"rca_multimodal_window_size_{window_size}.pkl")
            DatasetHandlerLikeOpenSource.split_and_save_dataset(
                modal_type_list=modal_types,
                modal_data=modal_data,
                ent_edge_index=edge_split,
                label_obj=label_obj,           # <-- 直接用你新 label pkl 的整包
                meta_data=meta,
                save_file_path=out_path,
                valid_ratio=self.cfg.valid_ratio,
                random_state=self.cfg.seed,
                shuffle=True,
                multi_class_label_format=True, # <-- 对齐开源用法
                num_of_fault_types=K,          # <-- 你是 5 类，就填 5（id=1..5, 0=normal）
                save_extra_fields=save_extra_fields,
                extra_keys=["entity_type", "template", "cmdb_id", "root_cause_type"],
            )


In [2]:
TEMP_DATA_STORAGE = "../../temp_data/aiops22"

node_list = ["node-1","node-2","node-3","node-4","node-5","node-6"]

service_list = [
    "adservice","cartservice","checkoutservice","currencyservice","emailservice",
    "frontend","paymentservice","productcatalogservice","recommendationservice","shippingservice"
]

pod_list = [
    'adservice-0','adservice-1','adservice-2','adservice2-0',
    'cartservice-0','cartservice-1','cartservice-2','cartservice2-0',
    'checkoutservice-0','checkoutservice-1','checkoutservice-2','checkoutservice2-0',
    'currencyservice-0','currencyservice-1','currencyservice-2','currencyservice2-0',
    'emailservice-0','emailservice-1','emailservice-2','emailservice2-0',
    'frontend-0','frontend-1','frontend-2','frontend2-0',
    'paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0',
    'productcatalogservice-0','productcatalogservice-1','productcatalogservice-2','productcatalogservice2-0',
    'recommendationservice-0','recommendationservice-1','recommendationservice-2','recommendationservice2-0',
    'shippingservice-0','shippingservice-1','shippingservice-2','shippingservice2-0'
]

all_entity_list = node_list + service_list + pod_list  # E=56

cfg = AIOpsDatasetGeneratorConfig(
    temp_data_storage=TEMP_DATA_STORAGE,
    window_size_list=[5, 10, 15],
    modal_types=["metric", "trace", "log"],
    valid_ratio=0.2,
    seed=409,

    node_list=node_list,
    service_list=service_list,
    pod_list=pod_list,
    all_entity_list=all_entity_list,

    # 你的 label 里 fault_type_id 是按这个列表映射成 1..K
    fault_type_list=[
        "node_cpu_failure",
        "node_disk_read_io_load",
        "node_memory_load",
        "node_disk_space_load",
        "node_disk_write_io_load",
        "node_cpu_spike",
        "container_network_packet_duplicate",
        "container_write_io_load",
        "container_read_io_load",
        "container_cpu_load",
        "container_network_latency",
        "container_process_abort",
        "container_network_packet_loss",
        "container_memory_load",
        "container_network_packet_corruption",
    ],
)

gen = AIOpsDatasetGeneratorOpenSourceLike(cfg)
gen.generate(save_extra_fields=True)


[GEN] window_size=5
[OK] saved -> ../../temp_data/aiops22/dataset/merge_multimodal/rca_multimodal_window_size_5.pkl
     y_train shape: (240, 56) (expect (N,E))

[GEN] window_size=10
[OK] saved -> ../../temp_data/aiops22/dataset/merge_multimodal/rca_multimodal_window_size_10.pkl
     y_train shape: (240, 56) (expect (N,E))

[GEN] window_size=15
[OK] saved -> ../../temp_data/aiops22/dataset/merge_multimodal/rca_multimodal_window_size_15.pkl
     y_train shape: (240, 56) (expect (N,E))
